In [2]:
import gzip
import json
import gc
from pathlib import Path
import duckdb

import pandas as pd
import numpy as np

# PHASE 3 : Kiểm tra chất lượng dữ liệu sau khi áp dụng ABSA và merge dữ liệu đa tiêu chí với dữ liệu reviews để phục vụ cho việc train mô hình

In [3]:
import gzip
import json
import gc
from pathlib import Path
import duckdb

import pandas as pd
import numpy as np

In [4]:
RAW_DIR = Path("../data/raw")
PROCESSED_DIR = Path("../data/processed")
META_TEMP_DIR = Path("../data/temp_metadata")
META_TEMP_DIR.mkdir(parents=True, exist_ok=True)

META_PATH = RAW_DIR / "meta_Electronics.json.gz"
FILTERED_PATH = PROCESSED_DIR / "reviews_filtered_min5.parquet"

TRAIN_PATH = PROCESSED_DIR / "train.parquet"
VAL_PATH = PROCESSED_DIR / "val.parquet"
TEST_PATH = PROCESSED_DIR / "test.parquet"

## STEP 1: Validate ABSA quality

In [4]:
import duckdb

con = duckdb.connect()

ABSA_PATH = Path("../data/processed/absa_train_chunks/*.parquet")

con.execute(f"""
SELECT
    COUNT(*) AS total_reviews,

    AVG(CASE WHEN crit_quality IS NOT NULL THEN 1 ELSE 0 END) * 100 AS quality_pct,

    AVG(CASE WHEN crit_value IS NOT NULL THEN 1 ELSE 0 END) * 100 AS value_pct,

    AVG(CASE WHEN crit_design IS NOT NULL THEN 1 ELSE 0 END) * 100 AS design_pct,

    AVG(CASE WHEN crit_usability IS NOT NULL THEN 1 ELSE 0 END) * 100 AS usability_pct,

    AVG(CASE WHEN crit_durability IS NOT NULL THEN 1 ELSE 0 END) * 100 AS durability_pct

FROM read_parquet('{ABSA_PATH}')
""").df()

,total_reviews,quality_pct,value_pct,design_pct,usability_pct,durability_pct
0,5677563,50.369217,24.306767,16.70745,34.292372,10.574889


## Step 2: Check aspect score distributions

In [9]:
con.execute(f"""
SELECT
    AVG(crit_quality) AS quality_mean,
    STDDEV(crit_quality) AS quality_std,

    AVG(crit_value) AS value_mean,
    STDDEV(crit_value) AS value_std,

    AVG(crit_design) AS design_mean,
    STDDEV(crit_design) AS design_std,

    AVG(crit_usability) AS usability_mean,
    STDDEV(crit_usability) AS usability_std,

    AVG(crit_durability) AS durability_mean,
    STDDEV(crit_durability) AS durability_std

FROM read_parquet('{ABSA_PATH}')
""").df()

,quality_mean,quality_std,value_mean,value_std,design_mean,design_std,usability_mean,usability_std,durability_mean,durability_std
0,3.914969,1.106511,3.767725,1.191427,3.759355,1.14321,3.477935,1.0619,3.111294,1.303233


## STEP 3 : Kiểm tra CORRELATION MATRIX

In [5]:
corr_df = con.execute(f"""
SELECT
    crit_quality,
    crit_value,
    crit_design,
    crit_usability,
    crit_durability
FROM read_parquet('{ABSA_PATH}')
""").df()

corr_df.corr()

,crit_quality,crit_value,crit_design,crit_usability,crit_durability
crit_quality,1.000000,0.409626,0.313960,0.373436,0.427826
crit_value,0.409626,1.000000,0.281081,0.288920,0.321763
crit_design,0.313960,0.281081,1.000000,0.308846,0.294582
crit_usability,0.373436,0.288920,0.308846,1.000000,0.361752
crit_durability,0.427826,0.321763,0.294582,0.361752,1.000000


## KIỂM TRA ITEM-LEVEL STABILITY

In [8]:
item_vectors = con.execute(f"""
SELECT
    item_id,

    AVG(crit_quality) AS quality,
    AVG(crit_value) AS value,
    AVG(crit_design) AS design,
    AVG(crit_usability) AS usability,
    AVG(crit_durability) AS durability,

    COUNT(*) AS n_reviews

FROM read_parquet('{ABSA_PATH}')

GROUP BY item_id

HAVING COUNT(*) >= 5

LIMIT 20
""").df()

item_vectors

,item_id,quality,value,design,usability,durability,n_reviews
0,B003E2RRZC,3.750237,NaN,2.959320,NaN,NaN,5
1,B003EB03OA,4.148371,3.593284,NaN,3.453917,3.144928,33
2,B003F41JGM,2.516877,2.794532,NaN,1.973007,NaN,6
3,B000N5CASW,3.078039,3.996980,4.890839,2.139941,NaN,5
4,B000N7VPZE,4.316477,4.154201,3.300598,3.536783,1.584473,12
5,B000N8RKIO,3.073415,3.093920,3.321926,3.486486,NaN,7
6,B000NDZ0O0,3.623820,3.996754,3.037180,3.257248,3.420538,22
7,B00005B8HO,3.817861,3.590175,2.453907,3.787963,4.827370,43
8,B00005BYER,3.927835,3.119688,3.817063,4.095365,3.165310,7
9,B00005NHH8,3.912504,3.224283,3.683416,2.943967,2.444336,30


## Step 4 : Build item-level criteria vectors

In [12]:
import duckdb
from pathlib import Path

con = duckdb.connect()

PROCESSED_DIR = Path("../data/processed")
ABSA_TRAIN_PATH = str(PROCESSED_DIR / "absa_train_chunks" / "*.parquet")
ITEM_CRITERIA_PATH = PROCESSED_DIR / "item_criteria_vectors_train.parquet"

con.execute(f"""
COPY (
    SELECT
        item_id,

        AVG(crit_quality) AS item_quality,
        AVG(crit_value) AS item_value,
        AVG(crit_design) AS item_design,
        AVG(crit_usability) AS item_usability,
        AVG(crit_durability) AS item_durability,

        COUNT(crit_quality) AS item_quality_count,
        COUNT(crit_value) AS item_value_count,
        COUNT(crit_design) AS item_design_count,
        COUNT(crit_usability) AS item_usability_count,
        COUNT(crit_durability) AS item_durability_count,

        COUNT(*) AS item_train_reviews

    FROM read_parquet('{ABSA_TRAIN_PATH}')
    GROUP BY item_id
) TO '{ITEM_CRITERIA_PATH}' (FORMAT PARQUET)
""")

In [13]:
# Kiểm tra coverage item criteria
con.execute(f"""
SELECT
    COUNT(*) AS items,

    AVG(CASE WHEN item_quality IS NOT NULL THEN 1 ELSE 0 END) * 100 AS quality_pct,
    AVG(CASE WHEN item_value IS NOT NULL THEN 1 ELSE 0 END) * 100 AS value_pct,
    AVG(CASE WHEN item_design IS NOT NULL THEN 1 ELSE 0 END) * 100 AS design_pct,
    AVG(CASE WHEN item_usability IS NOT NULL THEN 1 ELSE 0 END) * 100 AS usability_pct,
    AVG(CASE WHEN item_durability IS NOT NULL THEN 1 ELSE 0 END) * 100 AS durability_pct

FROM read_parquet('{ITEM_CRITERIA_PATH}')
""").df()

,items,quality_pct,value_pct,design_pct,usability_pct,durability_pct
0,270719,84.824117,67.718557,57.602902,74.182455,47.726979


## STEP 5  — Merge criteria vectors vào train set

In [2]:
import duckdb
from pathlib import Path

con = duckdb.connect()

PROCESSED_DIR = Path("../data/processed")

ITEM_CRITERIA_PATH = PROCESSED_DIR / "item_criteria_vectors_train.parquet"

splits = {
    "train": PROCESSED_DIR / "train.parquet",
    "val": PROCESSED_DIR / "val.parquet",
    "test": PROCESSED_DIR / "test.parquet",
    "val_warm": PROCESSED_DIR / "val_warm.parquet",
    "test_warm": PROCESSED_DIR / "test_warm.parquet",
}

In [15]:
for split_name, input_path in splits.items():
    output_path = PROCESSED_DIR / f"{split_name}_with_item_criteria.parquet"

    print(f"Merging {split_name}...")

    con.execute(f"""
    COPY (
        SELECT
            s.*,

            c.item_quality,
            c.item_value,
            c.item_design,
            c.item_usability,
            c.item_durability,

            c.item_quality_count,
            c.item_value_count,
            c.item_design_count,
            c.item_usability_count,
            c.item_durability_count,

            c.item_train_reviews

        FROM read_parquet('{input_path}') s
        LEFT JOIN read_parquet('{ITEM_CRITERIA_PATH}') c
            ON s.item_id = c.item_id

    ) TO '{output_path}' (FORMAT PARQUET)
    """)

    print(f"Saved: {output_path}")

Merging train...
Saved: ..\data\processed\train_with_item_criteria.parquet
Merging val...
Saved: ..\data\processed\val_with_item_criteria.parquet
Merging test...
Saved: ..\data\processed\test_with_item_criteria.parquet
Merging val_warm...
Saved: ..\data\processed\val_warm_with_item_criteria.parquet
Merging test_warm...
Saved: ..\data\processed\test_warm_with_item_criteria.parquet


In [3]:
# Kiểm tra sau khi merge
merged_check = con.execute(f"""
SELECT
    'train' AS split,
    COUNT(*) AS rows,
    AVG(CASE WHEN item_quality IS NOT NULL THEN 1 ELSE 0 END) * 100 AS quality_pct,
    AVG(CASE WHEN item_value IS NOT NULL THEN 1 ELSE 0 END) * 100 AS value_pct,
    AVG(CASE WHEN item_design IS NOT NULL THEN 1 ELSE 0 END) * 100 AS design_pct,
    AVG(CASE WHEN item_usability IS NOT NULL THEN 1 ELSE 0 END) * 100 AS usability_pct,
    AVG(CASE WHEN item_durability IS NOT NULL THEN 1 ELSE 0 END) * 100 AS durability_pct
FROM read_parquet('{PROCESSED_DIR / "train_with_item_criteria.parquet"}')

UNION ALL

SELECT
    'val',
    COUNT(*),
    AVG(CASE WHEN item_quality IS NOT NULL THEN 1 ELSE 0 END) * 100,
    AVG(CASE WHEN item_value IS NOT NULL THEN 1 ELSE 0 END) * 100,
    AVG(CASE WHEN item_design IS NOT NULL THEN 1 ELSE 0 END) * 100,
    AVG(CASE WHEN item_usability IS NOT NULL THEN 1 ELSE 0 END) * 100,
    AVG(CASE WHEN item_durability IS NOT NULL THEN 1 ELSE 0 END) * 100
FROM read_parquet('{PROCESSED_DIR / "val_with_item_criteria.parquet"}')

UNION ALL

SELECT
    'test',
    COUNT(*),
    AVG(CASE WHEN item_quality IS NOT NULL THEN 1 ELSE 0 END) * 100,
    AVG(CASE WHEN item_value IS NOT NULL THEN 1 ELSE 0 END) * 100,
    AVG(CASE WHEN item_design IS NOT NULL THEN 1 ELSE 0 END) * 100,
    AVG(CASE WHEN item_usability IS NOT NULL THEN 1 ELSE 0 END) * 100,
    AVG(CASE WHEN item_durability IS NOT NULL THEN 1 ELSE 0 END) * 100
FROM read_parquet('{PROCESSED_DIR / "test_with_item_criteria.parquet"}')

UNION ALL

SELECT
    'val_warm',
    COUNT(*),
    AVG(CASE WHEN item_quality IS NOT NULL THEN 1 ELSE 0 END) * 100,
    AVG(CASE WHEN item_value IS NOT NULL THEN 1 ELSE 0 END) * 100,
    AVG(CASE WHEN item_design IS NOT NULL THEN 1 ELSE 0 END) * 100,
    AVG(CASE WHEN item_usability IS NOT NULL THEN 1 ELSE 0 END) * 100,
    AVG(CASE WHEN item_durability IS NOT NULL THEN 1 ELSE 0 END) * 100
FROM read_parquet('{PROCESSED_DIR / "val_warm_with_item_criteria.parquet"}')

UNION ALL

SELECT
    'test_warm',
    COUNT(*),
    AVG(CASE WHEN item_quality IS NOT NULL THEN 1 ELSE 0 END) * 100,
    AVG(CASE WHEN item_value IS NOT NULL THEN 1 ELSE 0 END) * 100,
    AVG(CASE WHEN item_design IS NOT NULL THEN 1 ELSE 0 END) * 100,
    AVG(CASE WHEN item_usability IS NOT NULL THEN 1 ELSE 0 END) * 100,
    AVG(CASE WHEN item_durability IS NOT NULL THEN 1 ELSE 0 END) * 100
FROM read_parquet('{PROCESSED_DIR / "test_warm_with_item_criteria.parquet"}')
""").df()

merged_check

,split,rows,quality_pct,value_pct,design_pct,usability_pct,durability_pct
0,train,5677563,98.472496,95.343495,91.542251,96.714981,88.587216
1,val,975996,93.189214,87.446055,83.162329,89.884692,78.825118
2,test,405754,89.667631,82.869916,78.285119,85.970810,73.030210
3,val_warm,784495,96.462055,90.407969,86.021581,93.011810,81.497651
4,test_warm,305558,95.394655,88.062823,83.196644,91.449741,77.578725


In [6]:
val = pd.read_parquet(
    "../data/processed/val_with_item_criteria.parquet"
)

test = pd.read_parquet(
    "../data/processed/test_with_item_criteria.parquet"
)

In [1]:
import duckdb
import pandas as pd

pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)
pd.set_option('display.max_colwidth', None)

df = duckdb.query("""
    SELECT *
    FROM '../data/processed/model_ready/train_model.parquet'
    LIMIT 5
""").to_df()

print(df)

#################

   user_idx  item_idx  is_train_user  is_train_item  is_warm_row  \
0    214377     19573              1              1            1   
1    463188     19573              1              1            1   
2    349981     19573              1              1            1   
3    554534     19573              1              1            1   
4    348830     19573              1              1            1   

          user_id     item_id  rating  item_quality  item_value  item_design  \
0  A20YX6E89XZTQ0  B000BP8AY2     4.0      3.852027     3.73192     3.305106   
1  A381RVDWO44XLX  B000BP8AY2     4.0      3.852027     3.73192     3.305106   
2  A2OG33EMMWA39Z  B000BP8AY2     5.0      3.852027     3.73192     3.305106   
3  A3NT2DO5BKENZL  B000BP8AY2     5.0      3.852027     3.73192     3.305106   
4  A2O8TH8H4S3WS8  B000BP8AY2     5.0      3.852027     3.73192     3.305106   

   item_usability  item_durability  mask_quality  mask_value  mask_design  \
0        3.555008         2.67088

In [1]:
import pyarrow.parquet as pq
import numpy as np
import pandas as pd

MODEL_READY = "../data/processed/model_ready"

train_path = f"{MODEL_READY}/train_model.parquet"
val_path = f"{MODEL_READY}/val_model.parquet"
test_path = f"{MODEL_READY}/test_model.parquet"


def collect_unique_ids(path, col, batch_size=200_000):
    pf = pq.ParquetFile(path)
    ids = set()

    for batch in pf.iter_batches(columns=[col], batch_size=batch_size):
        arr = batch.column(0).to_numpy(zero_copy_only=False)

        # bỏ NaN nếu có
        arr = arr[~pd.isna(arr)]

        # convert sang int sau khi bỏ NaN
        ids.update(arr.astype(np.int64).tolist())

    return ids


def count_cold_rows(path, train_users, train_items, batch_size=200_000):
    pf = pq.ParquetFile(path)

    total_rows = 0

    nan_user_rows = 0
    nan_item_rows = 0

    cold_user_rows = 0
    cold_item_rows = 0

    warm_user_rows = 0
    warm_item_rows = 0

    for batch in pf.iter_batches(
        columns=["user_idx", "item_idx"],
        batch_size=batch_size,
    ):
        users = batch.column(0).to_numpy(zero_copy_only=False)
        items = batch.column(1).to_numpy(zero_copy_only=False)

        total_rows += len(users)

        user_nan_mask = pd.isna(users)
        item_nan_mask = pd.isna(items)

        nan_user_rows += int(user_nan_mask.sum())
        nan_item_rows += int(item_nan_mask.sum())

        valid_users = users[~user_nan_mask].astype(np.int64)
        valid_items = items[~item_nan_mask].astype(np.int64)

        # Nếu user_idx NaN => cold user
        cold_user_rows += int(user_nan_mask.sum())
        cold_item_rows += int(item_nan_mask.sum())

        # Nếu không NaN nhưng không thuộc train set => cũng cold
        cold_user_rows += sum(int(u) not in train_users for u in valid_users)
        cold_item_rows += sum(int(i) not in train_items for i in valid_items)

        warm_user_rows += sum(int(u) in train_users for u in valid_users)
        warm_item_rows += sum(int(i) in train_items for i in valid_items)

    return {
        "total_rows": total_rows,

        "nan_user_rows": nan_user_rows,
        "nan_item_rows": nan_item_rows,

        "cold_user_rows": cold_user_rows,
        "cold_item_rows": cold_item_rows,

        "warm_user_rows": warm_user_rows,
        "warm_item_rows": warm_item_rows,

        "cold_user_ratio": cold_user_rows / max(total_rows, 1),
        "cold_item_ratio": cold_item_rows / max(total_rows, 1),

        "nan_user_ratio": nan_user_rows / max(total_rows, 1),
        "nan_item_ratio": nan_item_rows / max(total_rows, 1),
    }


print("Collecting train users...")
train_users = collect_unique_ids(train_path, "user_idx")

print("Collecting train items...")
train_items = collect_unique_ids(train_path, "item_idx")

print("Train users:", len(train_users))
print("Train items:", len(train_items))

print("\nVAL")
val_stats = count_cold_rows(val_path, train_users, train_items)
print(val_stats)

print("\nTEST")
test_stats = count_cold_rows(test_path, train_users, train_items)
print(test_stats)

Train users: 752848
Train items: 270719

VAL
{'total_rows': 975996, 'nan_user_rows': 0, 'nan_item_rows': 0, 'cold_user_rows': 163202, 'cold_item_rows': 33225, 'warm_user_rows': 812794, 'warm_item_rows': 942771, 'cold_user_ratio': 0.1672158492452838, 'cold_item_ratio': 0.03404214771372014, 'nan_user_ratio': 0.0, 'nan_item_ratio': 0.0}

TEST
{'total_rows': 405754, 'nan_user_rows': 0, 'nan_item_rows': 0, 'cold_user_rows': 79767, 'cold_item_rows': 24583, 'warm_user_rows': 325987, 'warm_item_rows': 381171, 'cold_user_ratio': 0.19658955919103693, 'cold_item_ratio': 0.060585970809899595, 'nan_user_ratio': 0.0, 'nan_item_ratio': 0.0}
